In [2]:
import pandas as pd
import re
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
# from transformers import pipeline, Sentencr
from transformers import pipeline
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import warnings 
warnings.filterwarnings("ignore")
import numpy as np

from sentence_transformers import SentenceTransformer
df = pd.read_csv("D:\\Machine Learning\\Datasets\\netflix\\titles.csv")
df
crit = ( df['production_countries'].str.contains('IN')) & (df['type']=='MOVIE') & (df['release_year'] >= 1980)
df_ind_mov = df[crit]
df_ind_mov.head()
df_ind_mov.count()
df_ind_mov.loc[:, 'info'] = "Title: " + df_ind_mov['title'].copy() + " Description: " + df_ind_mov['description'].copy()
df_ind_mov.head()

print(df_ind_mov['info'].head())
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", token="hf_LJfVazBKDAmmoLJubhsJkMRSyaHaulEFcln")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
all_desc = list( df_ind_mov['info'])
out_embs = []
for r in tqdm(all_desc):
    embd= model.encode(r)
    out_embs.append(embd)
    
out_embs = np.array(out_embs)
out_embs.shape
df_ind_mov['movieID'] = df_ind_mov['id']+"-"+df_ind_mov['title']
df_embd = pd.DataFrame(out_embs, index=list(df_ind_mov['movieID']))
df_embd
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import silhouette_score
hdbscan = HDBSCAN(min_cluster_size=2)
hdbscan.fit(out_embs)
df_embd["hdbscan_cluster"] = hdbscan.labels_


19    Title: Dostana Description: Two close friends ...
56    Title: Vaashi Description: Ebin Mathew, a budd...
65    Title: Disco Dancer Description: Anil, a stree...
66    Title: Agneepath Description: The movie depict...
71    Title: Dil Description: Raja lives a poor life...
Name: info, dtype: object


100%|██████████| 570/570 [00:08<00:00, 69.62it/s]


In [6]:
!pip install bertopic

   ---------------------------------------- 0.0/687.1 kB ? eta -:--:--
   ---------------------------------------- 687.1/687.1 kB 4.6 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ---- ----------------------------------- 4.5/38.1 MB 22.4 MB/s eta 0:00:02
   --------- ------------------------------ 8.7/38.1 MB 20.7 MB/s eta 0:00:02
   -------------- ------------------------- 13.9/38.1 MB 21.8 MB/s eta 0:00:02
   ----------------- ---------------------- 17.0/38.1 MB 20.7 MB/s eta 0:00:02
   --------------------- ------------------ 21.0/38.1 MB 19.8 MB/s eta 0:00:01
   ------------------------- -------------- 23.9/38.1 MB 18.7 MB/s eta 0:00:01
   --------------------------- ------------ 26.2/38.1 MB 17.9 MB/s eta 0:00:01
   ----------------------------- ---------- 28.6/38.1 MB 16.8 MB/s eta 0:00:01
   -------------------------------- ------- 31.2/38.1 MB 16.5 MB/s eta 0:00:01
   ------------------------------------ --- 35.1/38.1 MB 16.7 MB/s eta 0:

In [7]:
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import silhouette_score
df_ind_mov['movieID'] = df_ind_mov['id']+'-'+df_ind_mov['title']
df_embd = pd.DataFrame(out_embs, index=list(df_ind_mov['movieID']))
df_embd
hdbscan = HDBSCAN(min_cluster_size=2)
hdbscan.fit(out_embs)
df_embd['hdbscan_cluster'] = hdbscan.labels_

df_embd_in = df_embd[df_embd["hdbscan_cluster"]!=-1]["hdbscan_cluster"]
print(df_embd_in[df_embd_in==54])
model = SentenceTransformer("all-MiniLM-L6-v2")
from bertopic import BERTopic
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(all_desc)
topic_model.get_topic_info()

tm831061-Magamuni    54
tm847212-Thambi      54
Name: hdbscan_cluster, dtype: int64


,Topic,Count,Name,Representation,Representative_Docs
0,-1,344,-1_the_to_and_of,"[the, to, and, of, in, title, description, his...",[Title: Chashme Baddoor Description: Siddharth...
1,0,81,0_love_and_to_description,"[love, and, to, description, title, in, her, w...",[Title: Theeya Velai Seiyyanum Kumaru Descript...
2,1,51,1_the_title_description_to,"[the, title, description, to, and, of, in, pol...",[Title: Kurup Description: The Film is based o...
3,2,41,2_his_to_father_title,"[his, to, father, title, description, the, he,...",[Title: Agneepath Description: The movie depic...
4,3,17,3_her_girl_description_title,"[her, girl, description, title, and, in, she, ...",[Title: Bulbul Can Sing Description: Coming-of...
5,4,13,4_the_to_singh_his,"[the, to, singh, his, is, and, with, of, in, for]",[Title: Hattrick Description: Despite cricket ...
6,5,13,5_blind_the_title_description,"[blind, the, title, description, and, he, of, ...",[Title: Barfi! Description: He is cute. He's s...
7,6,10,6_bheem_chhota_krishna_the,"[bheem, chhota, krishna, the, and, dholakpur, ...",[Title: Chhota Bheem and the ShiNobi Secret De...
